# Follow-Up Data Processing and RFS Endpoint Auditing

## Introduction

This notebook processes the TCGA-BRCA follow-up data to investigate recurrence and progression information and support the preparation of the recurrence-free survival (RFS) endpoint.

The main goal is to examine follow-up records, identify potential inconsistencies in event indicators and survival times, and assess the reliability of the patient-level outcome data before using it in the multimodal Transformer model.

## Main Steps

1. **Load the follow-up data:** Load the original `follow_up.tsv` file and inspect its structure, missing values, and duplicate records.
2. **Prepare patient IDs:** Extract patient IDs from the source data and standardize their format to support matching across datasets.
3. **Select relevant variables:** Keep the progression/recurrence indicator, days to recurrence, days to progression, and days to follow-up.
4. **Validate the source fields:** Check whether the required columns exist and examine the values recorded in the event and time fields.
5. **Normalize event indicators:** Categorize recorded values as Yes, No, missing, or unknown without automatically treating unknown values as negative events.
6. **Convert time fields:** Convert recurrence, progression, and follow-up times into numeric values while identifying missing, zero, negative, and invalid values.
7. **Perform row-level checks:** Identify potentially inconsistent records, including event indicators without positive event times, event times recorded alongside a No indicator, and event times later than the corresponding follow-up time.
8. **Perform patient-level auditing:** Group records by patient and investigate conflicting event indicators, multiple recorded event times, missing follow-up times, and other potential inconsistencies.
9. **Inspect additional source fields:** Examine other available follow-up variables and review the original records of patients flagged during the audit.
10. **Compare endpoint files:** Compare the previous processed endpoint with a newer candidate file to identify changes in patient coverage, event labels, and survival times.
11. **Investigate excluded patients:** Review patients missing from the newer candidate and inspect their original follow-up records to understand why they may have been excluded.
12. **Save audit reports:** Export row-level and patient-level validation results, flagged source records, and endpoint comparison reports for further review.

## Expected Outputs

The notebook generates audit files documenting potential data-quality issues, including:

- `followup_row_level_validation_audit.csv`
- `followup_patient_level_validation_audit.csv`
- `followup_additional_fields_audit.csv` (if additional fields are available)
- `followup_flagged_patients_source_records.csv`
- Endpoint comparison reports and records for patients requiring further investigation.

These files support the verification of the follow-up data and help identify problems that should be resolved before finalizing the RFS endpoint.

## Conclusion

This notebook focuses on **data validation and endpoint auditing**, rather than simply converting follow-up records into a final survival dataset. It investigates event indicators, recorded times, patient-level conflicts, and differences between endpoint versions.

The audit results help establish which patient records have usable and consistent outcome information. The final RFS endpoint should be finalized only after the relevant inconsistencies have been reviewed and the event and censoring rules have been clearly defined.

##Import Libraries

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
from google.colab import drive
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

print("Libraries loaded successfully.")

Libraries loaded successfully.


##Mount Google Drive

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

# drive.mount("/content/drive", force_remount=True)


drive.mount("/content/gdrive")
# drive.mount("/content/drive")

Mounted at /content/gdrive


##Load Follow-up Dataset

In [ ]:
# ============================================================
# LOAD FOLLOW-UP DATASET
# ============================================================

FOLLOWUP_PATH = "/content/gdrive/MyDrive/BreastCanser-DataSet/follow_up.tsv"

followup_raw = pd.read_csv(
    FOLLOWUP_PATH,
    sep="\t",
    low_memory=False
)

print("="*70)
print("FOLLOW-UP DATA LOADED")
print("="*70)

print("Shape:", followup_raw.shape)

FOLLOW-UP DATA LOADED
Shape: (9427, 152)


##Dataset Inspection

In [ ]:
# ============================================================
# DATASET INSPECTION
# ============================================================

print("="*70)
print("DATASET SUMMARY")
print("="*70)

print("\nRows:", followup_raw.shape[0])
print("Columns:", followup_raw.shape[1])

print("\nFirst five rows:")
display(followup_raw.head())

print("\nColumn names:")
print(list(followup_raw.columns[:20]))

print("\nMissing values:")
print(followup_raw.isna().sum().sum())

print("\nDuplicate rows:")
print(followup_raw.duplicated().sum())

DATASET SUMMARY

Rows: 9427
Columns: 152

First five rows:


,project.project_id,cases.case_id,cases.submitter_id,follow_ups.adverse_event,follow_ups.adverse_event_grade,follow_ups.barretts_esophagus_goblet_cells_present,follow_ups.cause_of_response,follow_ups.days_to_adverse_event,follow_ups.days_to_first_event,follow_ups.days_to_follow_up,follow_ups.days_to_imaging,follow_ups.days_to_progression,follow_ups.days_to_progression_free,follow_ups.days_to_recurrence,follow_ups.discontiguous_lesion_count,follow_ups.disease_response,follow_ups.ecog_performance_status,follow_ups.evidence_of_progression_type,follow_ups.evidence_of_recurrence_type,follow_ups.first_event,follow_ups.follow_up_id,follow_ups.histologic_progression,follow_ups.history_of_tumor,follow_ups.history_of_tumor_type,follow_ups.hormone_replacement_therapy_type,follow_ups.imaging_anatomic_site,follow_ups.imaging_findings,follow_ups.imaging_result,follow_ups.imaging_suv,follow_ups.imaging_suv_max,follow_ups.imaging_type,follow_ups.karnofsky_performance_status,follow_ups.peritoneal_washing_results,follow_ups.procedures_performed,follow_ups.progression_or_recurrence,follow_ups.progression_or_recurrence_anatomic_site,follow_ups.progression_or_recurrence_type,follow_ups.recist_targeted_regions_number,follow_ups.recist_targeted_regions_sum,follow_ups.scan_tracer_used,follow_ups.submitter_id,follow_ups.timepoint_category,follow_ups.treatment_emergent_adverse_event,follow_ups.year_of_follow_up,molecular_tests.aa_change,molecular_tests.aneuploidy,molecular_tests.antigen,molecular_tests.biospecimen_type,molecular_tests.biospecimen_volume,molecular_tests.blood_test_normal_range_lower,molecular_tests.blood_test_normal_range_upper,molecular_tests.cell_count,molecular_tests.chromosomal_translocation,molecular_tests.chromosome,molecular_tests.chromosome_arm,molecular_tests.clonality,molecular_tests.copy_number,molecular_tests.cytoband,molecular_tests.days_to_test,molecular_tests.exon,molecular_tests.gene_symbol,molecular_tests.histone_family,molecular_tests.histone_variant,molecular_tests.hpv_strain,molecular_tests.intron,molecular_tests.laboratory_test,molecular_tests.loci_abnormal_count,molecular_tests.loci_count,molecular_tests.locus,molecular_tests.mismatch_repair_mutation,molecular_tests.mitotic_count,molecular_tests.mitotic_total_area,molecular_tests.molecular_analysis_method,molecular_tests.molecular_consequence,molecular_tests.molecular_test_id,molecular_tests.mutation_codon,molecular_tests.pathogenicity,molecular_tests.ploidy,molecular_tests.second_exon,molecular_tests.second_gene_symbol,molecular_tests.specialized_molecular_test,molecular_tests.staining_intensity_scale,molecular_tests.staining_intensity_value,molecular_tests.submitter_id,molecular_tests.test_analyte_type,molecular_tests.test_result,molecular_tests.test_units,molecular_tests.test_value,molecular_tests.test_value_range,molecular_tests.timepoint_category,molecular_tests.transcript,molecular_tests.variant_origin,molecular_tests.variant_type,molecular_tests.zygosity,other_clinical_attributes.aids_risk_factors,other_clinical_attributes.bmi,other_clinical_attributes.body_surface_area,other_clinical_attributes.cd4_count,other_clinical_attributes.cdc_hiv_risk_factors,other_clinical_attributes.comorbidities,other_clinical_attributes.comorbidity_method_of_diagnosis,other_clinical_attributes.days_to_comorbidity,other_clinical_attributes.days_to_risk_factor,other_clinical_attributes.diabetes_treatment_type,other_clinical_attributes.dlco_ref_predictive_percent,other_clinical_attributes.exercise_frequency_weekly,other_clinical_attributes.eye_color,other_clinical_attributes.fertility_history,other_clinical_attributes.fev1_fvc_post_bronch_percent,other_clinical_attributes.fev1_fvc_pre_bronch_percent,other_clinical_attributes.fev1_ref_post_bronch_percent,other_clinical_attributes.fev1_ref_pre_bronch_percent,other_clinical_attributes.haart_treatment_indicator,other_clinical_attributes.height,other_clinical_attributes.hepatitis_sustained_virological_response,other_clinical_attributes.


Column names:
['project.project_id', 'cases.case_id', 'cases.submitter_id', 'follow_ups.adverse_event', 'follow_ups.adverse_event_grade', 'follow_ups.barretts_esophagus_goblet_cells_present', 'follow_ups.cause_of_response', 'follow_ups.days_to_adverse_event', 'follow_ups.days_to_first_event', 'follow_ups.days_to_follow_up', 'follow_ups.days_to_imaging', 'follow_ups.days_to_progression', 'follow_ups.days_to_progression_free', 'follow_ups.days_to_recurrence', 'follow_ups.discontiguous_lesion_count', 'follow_ups.disease_response', 'follow_ups.ecog_performance_status', 'follow_ups.evidence_of_progression_type', 'follow_ups.evidence_of_recurrence_type', 'follow_ups.first_event']

Missing values:
0

Duplicate rows:
0


##Patient ID Cleaning

In [ ]:
# ============================================================
# CREATE PATIENT ID
# ============================================================

print("="*70)
print("CREATING PATIENT IDENTIFIERS")
print("="*70)


# Rename dataframe
followup_df = followup_raw.copy()


# Extract patient ID
followup_df["patient_id"] = followup_df["cases.submitter_id"]


# Convert TCGA format

followup_df["patient_id"] = (
    followup_df["patient_id"]
    .astype(str)
    .str.replace(".", "-", regex=False)
)


print("Total follow-up rows:", len(followup_df))

print(
    "Unique patients:",
    followup_df["patient_id"].nunique()
)


print("\nExample IDs:")
print(
    followup_df["patient_id"]
    .head(10)
)

CREATING PATIENT IDENTIFIERS
Total follow-up rows: 9427
Unique patients: 1096

Example IDs:
0    TCGA-E2-A1IU
1    TCGA-E2-A1IU
2    TCGA-E2-A1IU
3    TCGA-E2-A1IU
4    TCGA-E2-A1IU
5    TCGA-E2-A1IU
6    TCGA-E2-A1IU
7    TCGA-E2-A1IU
8    TCGA-A1-A0SB
9    TCGA-A1-A0SB
Name: patient_id, dtype: object


##Keep Only Useful Columns

In [ ]:
# ============================================================
# KEEP REQUIRED FOLLOW-UP VARIABLES
# ============================================================

required_columns = [

    "patient_id",

    "follow_ups.progression_or_recurrence",

    "follow_ups.days_to_recurrence",

    "follow_ups.days_to_progression",

    "follow_ups.days_to_follow_up"

]


followup_df = followup_df[required_columns].copy()


print("="*70)
print("FOLLOW-UP VARIABLES SELECTED")
print("="*70)


print("Columns:")
print(followup_df.columns)


print("\nShape:")
print(followup_df.shape)


display(followup_df.head())

FOLLOW-UP VARIABLES SELECTED
Columns:
Index(['patient_id', 'follow_ups.progression_or_recurrence',
       'follow_ups.days_to_recurrence', 'follow_ups.days_to_progression',
       'follow_ups.days_to_follow_up'],
      dtype='object')

Shape:
(9427, 5)


,patient_id,follow_ups.progression_or_recurrence,follow_ups.days_to_recurrence,follow_ups.days_to_progression,follow_ups.days_to_follow_up
0,TCGA-E2-A1IU,'--,'--,'--,337
1,TCGA-E2-A1IU,'--,'--,'--,127
2,TCGA-E2-A1IU,'--,'--,'--,337
3,TCGA-E2-A1IU,'--,'--,'--,'--
4,TCGA-E2-A1IU,'--,'--,'--,'--


##Create and Save Audited RFS Endpoint

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

df = followup_df.copy()
df["patient_id"] = df["patient_id"].astype(str).str.strip()

def clean_flag(value):
    if pd.isna(value):
        return ""
    return str(value).strip().strip("'").lower()

def clean_time(series):
    values = (
        series.astype(str)
        .str.strip()
        .str.strip("'")
        .replace({
            "--": np.nan,
            "": np.nan,
            "nan": np.nan,
            "none": np.nan,
            "not reported": np.nan,
            "not available": np.nan,
        })
    )
    values = pd.to_numeric(values, errors="coerce")
    # Require strictly positive times; do not replace zero with one.
    return values.where(values > 0)

# Clean the source time fields.
time_cols = [
    "follow_ups.days_to_recurrence",
    "follow_ups.days_to_progression",
    "follow_ups.days_to_follow_up",
]

for col in time_cols:
    df[col] = clean_time(df[col])

flag_col = "follow_ups.progression_or_recurrence"
df["_explicit_event"] = df[flag_col].map(
    lambda x: clean_flag(x) in ["yes", "true", "1"]
)

# Earliest documented event time on each source record.
df["_event_time_row"] = df[
    [
        "follow_ups.days_to_recurrence",
        "follow_ups.days_to_progression",
    ]
].min(axis=1)

# Aggregate at patient level.
patient_rows = []

for patient_id, group in df.groupby("patient_id"):
    explicit_event = bool(group["_explicit_event"].any())

    event_times = group["_event_time_row"].dropna()
    followup_times = group["follow_ups.days_to_follow_up"].dropna()

    event_time = event_times.min() if not event_times.empty else np.nan
    last_followup = (
        followup_times.max() if not followup_times.empty else np.nan
    )

    if not pd.isna(event_time):
        # A valid recurrence/progression time exists.
        event = 1
        survival_time = float(event_time)
        status = "event_with_known_time"

    elif explicit_event:
        # Event reported, but its time is unknown: do not invent a time.
        event = np.nan
        survival_time = np.nan
        status = "event_time_unknown_excluded"

    elif not pd.isna(last_followup):
        # No event recorded; censor at last positive follow-up.
        event = 0
        survival_time = float(last_followup)
        status = "censored_with_valid_followup"

    else:
        event = np.nan
        survival_time = np.nan
        status = "no_valid_time_excluded"

    patient_rows.append({
        "patient_id": patient_id,
        "event": event,
        "survival_time": survival_time,
        "endpoint_status": status,
    })

audit = pd.DataFrame(patient_rows)

print("AUDIT SUMMARY")
print("Patients in raw follow-up:", len(audit))
print("\nEndpoint status counts:")
print(audit["endpoint_status"].value_counts(dropna=False))

eligible = audit[audit["event"].notna()].copy()
eligible["event"] = eligible["event"].astype(int)

print("\nEligible endpoint shape:", eligible.shape)
print("Unique patients:", eligible["patient_id"].nunique())
print("Event counts:")
print(eligible["event"].value_counts().sort_index())
print("Missing survival times:", eligible["survival_time"].isna().sum())
print("Non-positive survival times:", (eligible["survival_time"] <= 0).sum())

print("\nPatients with event time unknown:")
display(
    audit[audit["endpoint_status"] == "event_time_unknown_excluded"]
)

# Save new candidate files; existing endpoint files remain untouched.
out_dir = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint"
)
out_dir.mkdir(parents=True, exist_ok=True)

candidate_path = out_dir / "followup_rfs_audited_v2_candidate.csv"
audit_path = out_dir / "followup_rfs_audit_all_patients.csv"

eligible[["patient_id", "survival_time", "event"]].to_csv(
    candidate_path, index=False
)
audit.to_csv(audit_path, index=False)

print("\nSaved candidate:", candidate_path)
print("Saved full audit:", audit_path)

AUDIT SUMMARY
Patients in raw follow-up: 1096

Endpoint status counts:
endpoint_status
censored_with_valid_followup    979
event_with_known_time            83
no_valid_time_excluded           21
event_time_unknown_excluded      13
Name: count, dtype: int64

Eligible endpoint shape: (1062, 4)
Unique patients: 1062
Event counts:
event
0    979
1     83
Name: count, dtype: int64
Missing survival times: 0
Non-positive survival times: 0

Patients with event time unknown:


,patient_id,event,survival_time,endpoint_status
28,TCGA-A2-A04V,NaN,NaN,event_time_unknown_excluded
34,TCGA-A2-A0CM,NaN,NaN,event_time_unknown_excluded
68,TCGA-A2-A0SW,NaN,NaN,event_time_unknown_excluded
69,TCGA-A2-A0SX,NaN,NaN,event_time_unknown_excluded
173,TCGA-A8-A06Y,NaN,NaN,event_time_unknown_excluded
208,TCGA-A8-A08O,NaN,NaN,event_time_unknown_excluded
212,TCGA-A8-A08T,NaN,NaN,event_time_unknown_excluded
213,TCGA-A8-A08X,NaN,NaN,event_time_unknown_excluded
216,TCGA-A8-A091,NaN,NaN,event_time_unknown_excluded
399,TCGA-AR-A0TW,NaN,NaN,event_time_unknown_excluded



Saved candidate: /content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint/followup_rfs_audited_v2_candidate.csv
Saved full audit: /content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint/followup_rfs_audit_all_patients.csv


##Compare New RFS Endpoint with Existing Multimodal Dataset

In [ ]:

import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/TCGA_BRCA")

endpoint_path = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

master_path = base / "multimodal_rfs_dataset/multimodal_master_RFS_dataset.csv"

endpoint = pd.read_csv(endpoint_path)
master = pd.read_csv(master_path)

for df in [endpoint, master]:
    df["patient_id"] = df["patient_id"].astype(str).str.strip()

print("Audited endpoint patients:", endpoint["patient_id"].nunique())
print("Existing multimodal patients:", master["patient_id"].nunique())

# Match the existing multimodal cohort to the audited endpoint.
comparison = master[
    ["patient_id", "survival_time", "event"]
].merge(
    endpoint[["patient_id", "survival_time", "event"]],
    on="patient_id",
    how="left",
    suffixes=("_old", "_audited"),
    indicator=True,
)

print("\nMultimodal cohort matching:")
print(comparison["_merge"].value_counts())

matched = comparison[comparison["_merge"] == "both"].copy()

print("\nMatched patients:", len(matched))
print(
    "Event-label disagreements:",
    (matched["event_old"] != matched["event_audited"]).sum()
)
print(
    "Survival-time disagreements:",
    (
        (matched["survival_time_old"] - matched["survival_time_audited"])
        .abs() > 1e-8
    ).sum()
)

print("\nFirst 20 patients with changed survival times:")
display(
    matched.loc[
        (matched["survival_time_old"] - matched["survival_time_audited"])
        .abs() > 1e-8
    ].head(20)
)

print("\nPatients missing from the audited endpoint:")
display(
    comparison.loc[
        comparison["_merge"] == "left_only",
        ["patient_id", "survival_time_old", "event_old"]
    ].head(30)
)

Audited endpoint patients: 1062
Existing multimodal patients: 357

Multimodal cohort matching:
_merge
both          337
left_only      20
right_only      0
Name: count, dtype: int64

Matched patients: 337
Event-label disagreements: 0
Survival-time disagreements: 75

First 20 patients with changed survival times:


,patient_id,survival_time_old,event_old,survival_time_audited,event_audited,_merge
5,TCGA-AR-A0TY,1699.0,1,544.0,1.0,both
11,TCGA-AR-A24L,2866.0,1,1414.0,1.0,both
12,TCGA-C8-A137,379.0,1,136.0,1.0,both
14,TCGA-B6-A0IN,2573.0,1,1412.0,1.0,both
15,TCGA-LL-A5YM,394.0,1,258.0,1.0,both
23,TCGA-EW-A1P0,1251.0,1,1094.0,1.0,both
36,TCGA-A2-A0YC,990.0,1,756.0,1.0,both
47,TCGA-BH-A0C3,2709.0,1,2674.0,1.0,both
48,TCGA-B6-A0IE,1993.0,1,834.0,1.0,both
84,TCGA-B6-A0RH,6456.0,1,2165.0,1.0,both



Patients missing from the audited endpoint:


,patient_id,survival_time_old,event_old
34,TCGA-AC-A3W6,1.0,0
65,TCGA-A2-A04V,1920.0,1
79,TCGA-D8-A1XC,377.0,1
82,TCGA-AR-A24O,3607.0,1
105,TCGA-A8-A08X,1308.0,1
111,TCGA-C8-A26Y,1.0,0
117,TCGA-A8-A08T,3409.0,1
123,TCGA-A8-A08O,943.0,1
134,TCGA-A2-A0SX,1534.0,1
145,TCGA-A8-A06N,1.0,0


##Inspect Pathology Manifest and Search Notebooks

In [ ]:

import json
import pandas as pd
from pathlib import Path

drive = Path("/content/drive/MyDrive")
base = drive / "TCGA_BRCA"
notebook_dir = drive / "Colab Notebooks"

manifest_path = (
    base / "multimodal_rfs_dataset_leakage_free"
    / "pathology_512" / "pathology_512_manifest.csv"
)

metadata_path = base / "Features" / "features_metadata.csv"

notebooks_to_inspect = [
    "RFS_Transformer_Final_v4.ipynb",
    "RFS_Transformer_Final_final_v1.ipynb",
    "multimodal_rfs_dataset_splitting.ipynb",
    "multimodal_rfs_dataset_splitting (1).ipynb",
]

# ------------------------------------------------------------
# 1. Inspect the pathology manifest
# ------------------------------------------------------------
print("=" * 75)
print("1. PATHOLOGY MANIFEST")
print("=" * 75)

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)

    print("Path:", manifest_path)
    print("Shape:", manifest.shape)
    print("Columns:", manifest.columns.tolist())
    print("\nFirst five rows:")
    display(manifest.head())

    for col in manifest.columns:
        if any(term in col.lower() for term in [
            "patient", "file_uuid", "path", "patch", "mask", "slide"
        ]):
            print(
                f"{col}: {manifest[col].nunique(dropna=True)} unique, "
                f"{manifest[col].isna().sum()} missing"
            )
else:
    print("Manifest not found:", manifest_path)

# ------------------------------------------------------------
# 2. Inspect the pathology feature metadata
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("2. PATHOLOGY FEATURE METADATA")
print("=" * 75)

if metadata_path.exists():
    metadata = pd.read_csv(metadata_path)

    print("Path:", metadata_path)
    print("Shape:", metadata.shape)
    print("Columns:", metadata.columns.tolist())
    print("\nFirst five rows:")
    display(metadata.head())
else:
    print("Metadata file not found:", metadata_path)

# ------------------------------------------------------------
# 3. Search training/splitting notebooks for relevant code
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("3. RELEVANT NOTEBOOK CODE")
print("=" * 75)

keywords = [
    "PATHOLOGY_DIR",
    "pathology_512",
    "pathology_512_manifest",
    "features_metadata",
    "file_uuid",
    "PathologyEncoder",
    "MultimodalBreastCancerDataset",
    "pathology_features_path",
    "pathology_mask_path",
    "torch.load",
    "np.load",
    "random.sample",
    "randperm",
    "Cox",
    "train_rfs.csv",
    "multimodal_master_RFS_dataset.csv",
]

for notebook_name in notebooks_to_inspect:
    notebook_path = notebook_dir / notebook_name

    if not notebook_path.exists():
        print(f"\nNot found: {notebook_name}")
        continue

    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    print(f"\n\n### NOTEBOOK: {notebook_name}")

    found = 0

    for cell_idx, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        matched_lines = [
            i for i, line in enumerate(lines)
            if any(key.lower() in line.lower() for key in keywords)
        ]

        if not matched_lines:
            continue

        found += 1
        print(f"\n--- Cell {cell_idx} | {len(lines)} lines ---")

        # Print only matching lines and nearby context.
        selected = set()
        for i in matched_lines:
            selected.update(range(max(0, i - 2), min(len(lines), i + 3)))

        previous = -2
        for i in sorted(selected):
            if i > previous + 1:
                print("...")
            print(f"{i + 1:4}: {lines[i]}")
            previous = i

        if found >= 8:
            print("Stopped after 8 matching cells in this notebook.")
            break

    if found == 0:
        print("No matching code found.")

1. PATHOLOGY MANIFEST
Path: /content/drive/MyDrive/TCGA_BRCA/multimodal_rfs_dataset_leakage_free/pathology_512/pathology_512_manifest.csv
Shape: (357, 6)
Columns: ['patient_id', 'file_uuid', 'original_num_patches', 'selected_num_patches', 'padded_patches', 'feature_dim']

First five rows:


,patient_id,file_uuid,original_num_patches,selected_num_patches,padded_patches,feature_dim
0,TCGA-5L-AAT1,4eec69ca-381b-4c17-b3e9-49492d71560e,1777,512,0,512
1,TCGA-A1-A0SB,cea82b7d-135a-49d5-b4f6-3fb0215f7188,4512,512,0,512
2,TCGA-A1-A0SF,f273a858-1cfa-4983-ace4-b1e876d64473,6094,512,0,512
3,TCGA-A1-A0SI,a951559a-e6c0-419b-b09f-43e6cb6ee3ea,6822,512,0,512
4,TCGA-A1-A0SJ,9a3a3830-9674-4f9b-985f-e50062e78807,4801,512,0,512


patient_id: 357 unique, 0 missing
file_uuid: 357 unique, 0 missing
original_num_patches: 353 unique, 0 missing
selected_num_patches: 1 unique, 0 missing
padded_patches: 19 unique, 0 missing

2. PATHOLOGY FEATURE METADATA
Path: /content/drive/MyDrive/TCGA_BRCA/Features/features_metadata.csv
Shape: (369, 7)
Columns: ['patient_id', 'num_patches', 'feature_shape', 'num_patches_total', 'num_patches_accepted', 'num_patches_rejected', 'num_errors']

First five rows:


,patient_id,num_patches,feature_shape,num_patches_total,num_patches_accepted,num_patches_rejected,num_errors
0,a1b58cd0-559e-467d-a193-96602cd3306f,12697.0,"(12697, 512)",NaN,NaN,NaN,NaN
1,997dbd8f-9f16-4fc2-8627-d29ee0d74ebe,1989.0,"(1989, 512)",NaN,NaN,NaN,NaN
2,9132985f-ecd8-4a67-ba7b-63902801fba6,9047.0,"(9047, 512)",NaN,NaN,NaN,NaN
3,ce77af22-10b7-4480-9c93-015970268455,13609.0,"(13609, 512)",NaN,NaN,NaN,NaN
4,fcc84daa-5e54-4773-88a8-aa76727ffb6c,10695.0,"(10695, 512)",NaN,NaN,NaN,NaN



3. RELEVANT NOTEBOOK CODE


### NOTEBOOK: RFS_Transformer_Final_v4.ipynb

--- Cell 2 | 14 lines ---
...
   9: 
  10: from sksurv.svm import FastSurvivalSVM
  11: from sksurv.linear_model import CoxnetSurvivalAnalysis
  12: 
  13: print("FastSurvivalSVM: OK")
  14: print("CoxnetSurvivalAnalysis: OK")

--- Cell 7 | 9 lines ---
...
   4: 
   5: from sksurv.svm import FastSurvivalSVM
   6: from sksurv.linear_model import CoxnetSurvivalAnalysis
   7: 
   8: print("FastSurvivalSVM: OK")
   9: print("CoxnetSurvivalAnalysis: OK")

--- Cell 12 | 60 lines ---
...
  15: 
  16: # Final pathology tensors + masks
  17: PATHOLOGY_DIR = os.path.join(
  18:     DATASET_DIR,
  19:     "pathology_512"
  20: )
  21: 
...
  52: 
  53: print("\nPathology directory:")
  54: print(PATHOLOGY_DIR)
  55: 
  56: print("\nModel directory:")

--- Cell 15 | 100 lines ---
...
  12: 
  13:     "patient_id",
  14:     "file_uuid",
  15:     "survival_time",
  16:     "event",
  17:     "split",
  18:     "pathology_fe

##Inspect Transformer Model and Training Code

In [ ]:

import json
from pathlib import Path

NOTEBOOK_DIR = Path("/content/drive/MyDrive/Colab Notebooks")

NOTEBOOKS = [
    NOTEBOOK_DIR / "RFS_Transformer_Final_v4.ipynb",
    NOTEBOOK_DIR / "RFS_Transformer_Final_final_v1.ipynb",
]

KEYWORDS = [
    "class Multimodal",
    "def forward",
    "pathology_encoder(",
    '["mask"]',
    "PATHOLOGY_DIR",
    "pathology_512",
    "pathology_features_path",
    "src_key_padding_mask",
    "cox",
    "checkpoint",
    "torch.save",
    "torch.load",
]

for notebook_path in NOTEBOOKS:
    print("\n" + "=" * 90)
    print("NOTEBOOK:", notebook_path.name)
    print("=" * 90)

    if not notebook_path.exists():
        print("NOT FOUND:", notebook_path)
        continue

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    matches = []

    for i, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        if any(k.lower() in source.lower() for k in KEYWORDS):
            matches.append((i, source))

    print("Matching cells:", len(matches))

    for i, source in matches:
        print(f"\n--- Cell {i} ---")
        print(source[:9000])
        if len(source) > 9000:
            print("\n[Cell truncated for display]")


NOTEBOOK: RFS_Transformer_Final_v4.ipynb
Matching cells: 40

--- Cell 2 ---
!pip install scikit-survival
import sys
import sklearn
import sksurv

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("scikit-survival:", sksurv.__version__)

from sksurv.svm import FastSurvivalSVM
from sksurv.linear_model import CoxnetSurvivalAnalysis

print("FastSurvivalSVM: OK")
print("CoxnetSurvivalAnalysis: OK")

--- Cell 7 ---
import sksurv

print("scikit-survival:", sksurv.__version__)

from sksurv.svm import FastSurvivalSVM
from sksurv.linear_model import CoxnetSurvivalAnalysis

print("FastSurvivalSVM: OK")
print("CoxnetSurvivalAnalysis: OK")

--- Cell 12 ---
# ============================================================
# FINAL LEAKAGE-FREE PROJECT PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/TCGA_BRCA"

# ------------------------------------------------------------
# FINAL MULTIMODAL DATASET
# ----------------

##Investigation code:

In [ ]:

# ============================================================
# RFS ENDPOINT AUDIT — PART A
# Inspect source fields and validate their values
# Does NOT create or overwrite the final endpoint
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

FOLLOWUP_PATH = (
    "/content/gdrive/MyDrive/BreastCanser-DataSet/follow_up.tsv"
)

AUDIT_DIR = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint"
)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Reload the original source so this audit is reproducible.
raw = pd.read_csv(
    FOLLOWUP_PATH,
    sep="\t",
    low_memory=False
)

ID_COL = "cases.submitter_id"
FLAG_COL = "follow_ups.progression_or_recurrence"
RECURRENCE_COL = "follow_ups.days_to_recurrence"
PROGRESSION_COL = "follow_ups.days_to_progression"
FOLLOWUP_COL = "follow_ups.days_to_follow_up"

required = [
    ID_COL,
    FLAG_COL,
    RECURRENCE_COL,
    PROGRESSION_COL,
    FOLLOWUP_COL,
]

missing_columns = [c for c in required if c not in raw.columns]

if missing_columns:
    raise ValueError(
        "Required columns are missing from the source file: "
        + str(missing_columns)
    )

print("=" * 75)
print("SOURCE DATA VALIDATION")
print("=" * 75)

print("Source path:", FOLLOWUP_PATH)
print("Raw shape:", raw.shape)
print("Required columns found:", len(required), "/", len(required))
print("Raw rows:", len(raw))

# ------------------------------------------------------------
# 1. Inspect the time fields exactly as they occur in the source
# ------------------------------------------------------------

for col in [RECURRENCE_COL, PROGRESSION_COL, FOLLOWUP_COL]:
    print("\n" + "-" * 65)
    print("FIELD:", col)
    print("Raw values (including missing):")
    print(raw[col].astype("string").value_counts(dropna=False).head(20))

# ------------------------------------------------------------
# 2. Normalize patient IDs without turning missing IDs into "nan"
# ------------------------------------------------------------

df = raw[required].copy()

df["patient_id"] = (
    df[ID_COL]
    .astype("string")
    .str.strip()
    .str.replace(".", "-", regex=False)
)

invalid_id = (
    df["patient_id"].isna()
    | df["patient_id"].eq("")
    | df["patient_id"].str.lower().eq("nan")
)

print("\nRows with missing/invalid patient ID:", int(invalid_id.sum()))
print("Unique non-missing patient IDs:",
      df.loc[~invalid_id, "patient_id"].nunique())

# ------------------------------------------------------------
# 3. Parse flags without silently assuming unknown means "No"
# ------------------------------------------------------------

def normalize_flag(value):
    if pd.isna(value):
        return "missing"

    value = str(value).strip().strip("'").lower()

    if value in {"yes", "true", "1"}:
        return "yes"
    if value in {"no", "false", "0"}:
        return "no"
    if value in {"", "--", "nan", "none", "not reported",
                 "not available", "unknown"}:
        return "unknown"

    return "other:" + value

df["flag_normalized"] = df[FLAG_COL].map(normalize_flag)

print("\nNormalized progression/recurrence flag counts:")
print(df["flag_normalized"].value_counts(dropna=False))

# ------------------------------------------------------------
# 4. Parse time fields but retain zero and negative values
#    so that they can be audited rather than silently discarded.
# ------------------------------------------------------------

def parse_time(series):
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.strip("'")
        .replace({
            "--": pd.NA,
            "": pd.NA,
            "nan": pd.NA,
            "none": pd.NA,
            "not reported": pd.NA,
            "not available": pd.NA,
            "unknown": pd.NA,
        })
    )
    return pd.to_numeric(cleaned, errors="coerce")

for col in [RECURRENCE_COL, PROGRESSION_COL, FOLLOWUP_COL]:
    df[col + "_numeric"] = parse_time(df[col])

    parsed = df[col + "_numeric"]
    print(f"\n{col}")
    print("  Missing/unparseable:", int(parsed.isna().sum()))
    print("  Zero:", int((parsed == 0).sum()))
    print("  Negative:", int((parsed < 0).sum()))
    print("  Positive:", int((parsed > 0).sum()))

# ------------------------------------------------------------
# 5. Row-level flags for suspicious combinations
# ------------------------------------------------------------

rec = df[RECURRENCE_COL + "_numeric"]
prog = df[PROGRESSION_COL + "_numeric"]
fu = df[FOLLOWUP_COL + "_numeric"]

df["row_has_positive_event_time"] = (
    rec.gt(0) | prog.gt(0)
)

df["row_has_nonpositive_event_time"] = (
    rec.le(0).fillna(False) | prog.le(0).fillna(False)
)

df["row_has_positive_followup"] = fu.gt(0)

df["row_flag_yes_without_positive_event_time"] = (
    df["flag_normalized"].eq("yes")
    & ~df["row_has_positive_event_time"]
)

df["row_flag_no_with_positive_event_time"] = (
    df["flag_normalized"].eq("no")
    & df["row_has_positive_event_time"]
)

# A recorded event time later than the follow-up time on the same row
# is suspicious, but does not by itself prove which field is wrong.
row_event_time = pd.concat([rec, prog], axis=1).min(axis=1)

df["row_event_after_followup"] = (
    row_event_time.notna()
    & fu.notna()
    & row_event_time.gt(fu)
)

print("\n" + "=" * 75)
print("ROW-LEVEL QUALITY CHECKS")
print("=" * 75)

checks = [
    "row_has_positive_event_time",
    "row_has_nonpositive_event_time",
    "row_has_positive_followup",
    "row_flag_yes_without_positive_event_time",
    "row_flag_no_with_positive_event_time",
    "row_event_after_followup",
]

for col in checks:
    print(f"{col}: {int(df[col].sum())}")

# Save a row-level audit with original values plus parsed values/flags.
row_audit_path = AUDIT_DIR / "followup_row_level_validation_audit.csv"
df.to_csv(row_audit_path, index=False)

print("\nSaved row-level audit:")
print(row_audit_path)


SOURCE DATA VALIDATION
Source path: /content/gdrive/MyDrive/BreastCanser-DataSet/follow_up.tsv
Raw shape: (9427, 152)
Required columns found: 5 / 5
Raw rows: 9427

-----------------------------------------------------------------
FIELD: follow_ups.days_to_recurrence
Raw values (including missing):
follow_ups.days_to_recurrence
'--     9398
596        2
136        1
181        1
155        1
301        1
746        1
1516       1
2330       1
1645       1
648        1
1034       1
3287       1
431        1
756        1
966        1
667        1
419        1
1709       1
2989       1
Name: count, dtype: Int64

-----------------------------------------------------------------
FIELD: follow_ups.days_to_progression
Raw values (including missing):
follow_ups.days_to_progression
'--     9340
301        4
1351       2
443        2
524        2
192        2
188        2
790        2
1412       1
2717       1
729        1
859        1
489        1
396        1
536        1
207        1
1922     

##Check conflicts across all records for each patient

In [ ]:

# ============================================================
# RFS ENDPOINT AUDIT — PART B
# Patient-level conflicts and missing information
# Does NOT assign a final event/time endpoint
# ============================================================

valid_rows = df.loc[~invalid_id].copy()

def distinct_positive_values(series):
    values = sorted(series[series > 0].dropna().unique().tolist())
    return values

patient_audit_rows = []

for patient_id, group in valid_rows.groupby("patient_id", sort=True):

    flags = set(group["flag_normalized"].tolist())

    rec_times = distinct_positive_values(
        group[RECURRENCE_COL + "_numeric"]
    )
    prog_times = distinct_positive_values(
        group[PROGRESSION_COL + "_numeric"]
    )
    followup_times = distinct_positive_values(
        group[FOLLOWUP_COL + "_numeric"]
    )

    all_event_times = sorted(set(rec_times + prog_times))

    has_yes = "yes" in flags
    has_no = "no" in flags

    issues = []

    if has_yes and has_no:
        issues.append("conflicting_yes_and_no_flags")

    if has_yes and not all_event_times:
        issues.append("event_flag_yes_but_no_positive_event_time")

    if has_no and all_event_times:
        issues.append("event_flag_no_but_positive_event_time_exists")

    if len(rec_times) > 1:
        issues.append("multiple_distinct_recurrence_times")

    if len(prog_times) > 1:
        issues.append("multiple_distinct_progression_times")

    if len(all_event_times) > 1:
        issues.append("multiple_distinct_event_times")

    if not followup_times:
        issues.append("no_positive_followup_time")

    if all_event_times and followup_times:
        earliest_event = min(all_event_times)
        latest_followup = max(followup_times)

        if earliest_event > latest_followup:
            issues.append("earliest_event_after_latest_followup")
    else:
        earliest_event = (
            min(all_event_times) if all_event_times else np.nan
        )
        latest_followup = (
            max(followup_times) if followup_times else np.nan
        )

    if not all_event_times and not has_yes and not followup_times:
        issues.append("no_event_time_and_no_followup_time")

    if not all_event_times and has_yes:
        potential_status = "event_reported_time_unresolved"
    elif all_event_times:
        potential_status = "event_time_recorded_requires_review"
    elif followup_times and not has_yes:
        potential_status = "possible_censoring_requires_review"
    else:
        potential_status = "unresolved"

    patient_audit_rows.append({
        "patient_id": patient_id,
        "number_of_followup_rows": len(group),
        "flag_values": ";".join(sorted(flags)),
        "has_yes_flag": has_yes,
        "has_no_flag": has_no,
        "recurrence_times_positive_days": ";".join(map(str, rec_times)),
        "progression_times_positive_days": ";".join(map(str, prog_times)),
        "all_event_times_positive_days": ";".join(map(str, all_event_times)),
        "followup_times_positive_days": ";".join(map(str, followup_times)),
        "earliest_recorded_event_time_days": earliest_event,
        "latest_recorded_followup_time_days": latest_followup,
        "number_of_distinct_event_times": len(all_event_times),
        "number_of_distinct_followup_times": len(followup_times),
        "potential_status_not_final": potential_status,
        "issues": ";".join(issues) if issues else "none_detected",
    })

patient_audit = pd.DataFrame(patient_audit_rows)

print("=" * 75)
print("PATIENT-LEVEL AUDIT SUMMARY")
print("=" * 75)

print("Patients audited:", len(patient_audit))
print("Patients with any flagged issue:",
      int(patient_audit["issues"].ne("none_detected").sum()))

print("\nPotential status counts:")
print(patient_audit["potential_status_not_final"].value_counts(
    dropna=False
))

print("\nIssue counts (a patient can have multiple issues):")
issue_counts = (
    patient_audit.loc[
        patient_audit["issues"].ne("none_detected"), "issues"
    ]
    .str.split(";")
    .explode()
    .value_counts()
)
print(issue_counts)

patient_audit_path = AUDIT_DIR / "followup_patient_level_validation_audit.csv"
patient_audit.to_csv(patient_audit_path, index=False)

print("\nSaved patient-level audit:")
print(patient_audit_path)

print("\nFirst 30 patients requiring review:")
display(
    patient_audit.loc[
        patient_audit["issues"].ne("none_detected")
    ].head(30)
)


PATIENT-LEVEL AUDIT SUMMARY
Patients audited: 1096
Patients with any flagged issue: 49

Potential status counts:
potential_status_not_final
possible_censoring_requires_review     979
event_time_recorded_requires_review     83
unresolved                              21
event_reported_time_unresolved          13
Name: count, dtype: int64

Issue counts (a patient can have multiple issues):
issues
no_event_time_and_no_followup_time           21
no_positive_followup_time                    21
multiple_distinct_event_times                15
event_flag_yes_but_no_positive_event_time    13
multiple_distinct_progression_times          10
multiple_distinct_recurrence_times            3
Name: count, dtype: int64

Saved patient-level audit:
/content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint/followup_patient_level_validation_audit.csv

First 30 patients requiring review:


,patient_id,number_of_followup_rows,flag_values,has_yes_flag,has_no_flag,recurrence_times_positive_days,progression_times_positive_days,all_event_times_positive_days,followup_times_positive_days,earliest_recorded_event_time_days,latest_recorded_followup_time_days,number_of_distinct_event_times,number_of_distinct_followup_times,potential_status_not_final,issues
23,TCGA-A2-A04P,18,unknown;yes,True,False,179,102;180;313,102;179;180;313,102;179;180;313;548,102.0,548.0,4,5,event_time_recorded_requires_review,multiple_distinct_progression_times;multiple_d...
28,TCGA-A2-A04V,10,unknown;yes,True,False,,,,1470;1920,NaN,1920.0,0,2,event_reported_time_unresolved,event_flag_yes_but_no_positive_event_time
34,TCGA-A2-A0CM,8,unknown;yes,True,False,,,,382;754,NaN,754.0,0,2,event_reported_time_unresolved,event_flag_yes_but_no_positive_event_time
68,TCGA-A2-A0SW,9,unknown;yes,True,False,,,,653;1365,NaN,1365.0,0,2,event_reported_time_unresolved,event_flag_yes_but_no_positive_event_time
69,TCGA-A2-A0SX,9,unknown;yes,True,False,,,,1239;1288;1534,NaN,1534.0,0,3,event_reported_time_unresolved,event_flag_yes_but_no_positive_event_time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
252,TCGA-AC-A23H,9,unknown,False,False,,,,,NaN,NaN,0,0,unresolved,no_positive_followup_time;no_event_time_and_no...
257,TCGA-AC-A2FE,10,unknown;yes,True,False,,766;2496,766;2496,766;791;2240;2496;2636,766.0,2636.0,2,5,event_time_recorded_requires_review,multiple_distinct_progression_times;multiple_d...
275,TCGA-AC-A3W6,7,unknown,False,False,,,,,NaN,NaN,0,0,unresolved,no_positive_followup_time;no_event_time_and_no...
399,TCGA-AR-A0TW,11,unknown;yes,True,False,,,,739;1966;3009,NaN,3009.0,0,3,event_reported_time_unresolved,event_flag_yes_but_no_positive_event_time


##Inspect additional fields that may clarify the endpoint

In [ ]:

# ============================================================
# RFS ENDPOINT AUDIT — PART C
# Inspect other potentially relevant source fields
# ============================================================

possible_extra_fields = [
    "follow_ups.days_to_first_event",
    "follow_ups.first_event",
    "follow_ups.progression_or_recurrence_type",
    "follow_ups.evidence_of_recurrence_type",
    "follow_ups.evidence_of_progression_type",
    "follow_ups.days_to_progression_free",
    "follow_ups.timepoint_category",
    "follow_ups.year_of_follow_up",
    "follow_ups.submitter_id",
]

available_extra_fields = [
    col for col in possible_extra_fields
    if col in raw.columns
]

missing_extra_fields = [
    col for col in possible_extra_fields
    if col not in raw.columns
]

print("Additional fields available:")
print(available_extra_fields)

print("\nAdditional fields not present:")
print(missing_extra_fields)

for col in available_extra_fields:
    print("\n" + "=" * 70)
    print("FIELD:", col)
    print("Non-missing values:", int(raw[col].notna().sum()))
    print(raw[col].astype("string").value_counts(
        dropna=False
    ).head(20))

if available_extra_fields:
    extra_audit_path = (
        AUDIT_DIR / "followup_additional_fields_audit.csv"
    )

    raw[
        [ID_COL] + available_extra_fields
    ].to_csv(extra_audit_path, index=False)

    print("\nSaved additional-fields audit:")
    print(extra_audit_path)


Additional fields available:
['follow_ups.days_to_first_event', 'follow_ups.first_event', 'follow_ups.progression_or_recurrence_type', 'follow_ups.evidence_of_recurrence_type', 'follow_ups.evidence_of_progression_type', 'follow_ups.days_to_progression_free', 'follow_ups.timepoint_category', 'follow_ups.year_of_follow_up', 'follow_ups.submitter_id']

Additional fields not present:
[]

FIELD: follow_ups.days_to_first_event
Non-missing values: 9427
follow_ups.days_to_first_event
'--    9427
Name: count, dtype: Int64

FIELD: follow_ups.first_event
Non-missing values: 9427
follow_ups.first_event
'--    9427
Name: count, dtype: Int64

FIELD: follow_ups.progression_or_recurrence_type
Non-missing values: 9427
follow_ups.progression_or_recurrence_type
'--             9286
Distant           89
Locoregional      30
Unknown           22
Name: count, dtype: Int64

FIELD: follow_ups.evidence_of_recurrence_type
Non-missing values: 9427
follow_ups.evidence_of_recurrence_type
'--    9427
Name: count, d

In [ ]:

# ============================================================
# RFS ENDPOINT AUDIT — PART D
# Inspect the actual source records for flagged patients
# Does NOT modify or overwrite any endpoint
# ============================================================

import pandas as pd
from pathlib import Path

review_dir = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint"
)

# Source columns that help explain each follow-up record.
review_columns = [
    "cases.submitter_id",
    "follow_ups.submitter_id",
    "follow_ups.timepoint_category",
    "follow_ups.progression_or_recurrence",
    "follow_ups.progression_or_recurrence_type",
    "follow_ups.days_to_recurrence",
    "follow_ups.days_to_progression",
    "follow_ups.days_to_follow_up",
]

review_columns = [
    col for col in review_columns if col in raw.columns
]

review = raw[review_columns].copy()

review["patient_id"] = (
    review["cases.submitter_id"]
    .astype("string")
    .str.strip()
    .str.replace(".", "-", regex=False)
)

# Find patients with potential issues from the earlier audit.
flagged = patient_audit[
    patient_audit["issues"].ne("none_detected")
].copy()

flagged_ids = set(flagged["patient_id"].astype(str))

review_flagged = review[
    review["patient_id"].astype(str).isin(flagged_ids)
].copy()

print("Flagged patients:", len(flagged_ids))
print("Source records belonging to flagged patients:",
      len(review_flagged))

# Show the actual longitudinal records, not only the summary.
display(
    review_flagged.sort_values(
        ["patient_id", "follow_ups.submitter_id"],
        na_position="last"
    ).head(100)
)

# Export all records for flagged patients for easier inspection.
review_path = review_dir / "followup_flagged_patients_source_records.csv"
review_flagged.to_csv(review_path, index=False)

print("\nSaved all source records for flagged patients:")
print(review_path)

# ------------------------------------------------------------
# Focus on the 13 patients with a YES flag but no positive
# recurrence/progression time.
# ------------------------------------------------------------

unknown_event_ids = set(
    patient_audit.loc[
        patient_audit["issues"].str.contains(
            "event_flag_yes_but_no_positive_event_time",
            na=False
        ),
        "patient_id"
    ].astype(str)
)

print("\nPatients with event flag YES but no positive event time:",
      len(unknown_event_ids))

display(
    review[review["patient_id"].astype(str).isin(
        unknown_event_ids
    )].sort_values(
        ["patient_id", "follow_ups.submitter_id"],
        na_position="last"
    )
)

# ------------------------------------------------------------
# Summarize the record categories and event types.
# ------------------------------------------------------------

for col in [
    "follow_ups.timepoint_category",
    "follow_ups.progression_or_recurrence_type",
]:
    if col in review.columns:
        print("\n" + "=" * 70)
        print(col)
        print(
            review[col]
            .astype("string")
            .str.strip()
            .value_counts(dropna=False)
        )


Flagged patients: 49
Source records belonging to flagged patients: 520


,cases.submitter_id,follow_ups.submitter_id,follow_ups.timepoint_category,follow_ups.progression_or_recurrence,follow_ups.progression_or_recurrence_type,follow_ups.days_to_recurrence,follow_ups.days_to_progression,follow_ups.days_to_follow_up,patient_id
7572,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04P
7573,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04P
7574,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04P
7575,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04P
7576,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04P
...,...,...,...,...,...,...,...,...,...
6361,TCGA-A2-A3XU,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A3XU
6362,TCGA-A2-A3XU,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A3XU
6363,TCGA-A2-A3XU,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A3XU
6343,TCGA-A2-A3XU,TCGA-A2-A3XU_follow_up,Last Contact,'--,'--,'--,'--,912,TCGA-A2-A3XU



Saved all source records for flagged patients:
/content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint/followup_flagged_patients_source_records.csv

Patients with event flag YES but no positive event time: 13


,cases.submitter_id,follow_ups.submitter_id,follow_ups.timepoint_category,follow_ups.progression_or_recurrence,follow_ups.progression_or_recurrence_type,follow_ups.days_to_recurrence,follow_ups.days_to_progression,follow_ups.days_to_follow_up,patient_id
3702,TCGA-A2-A04V,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04V
3703,TCGA-A2-A04V,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04V
3704,TCGA-A2-A04V,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04V
3705,TCGA-A2-A04V,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04V
3706,TCGA-A2-A04V,'--,'--,'--,'--,'--,'--,'--,TCGA-A2-A04V
...,...,...,...,...,...,...,...,...,...
8855,TCGA-D8-A1XC,'--,'--,'--,'--,'--,'--,'--,TCGA-D8-A1XC
8848,TCGA-D8-A1XC,TCGA-D8-A1XC_follow_up,Last Contact,'--,'--,'--,'--,377,TCGA-D8-A1XC
8847,TCGA-D8-A1XC,TCGA-D8-A1XC_follow_up2,Follow-up,'--,'--,'--,'--,81,TCGA-D8-A1XC
8849,TCGA-D8-A1XC,TCGA-D8-A1XC_follow_up3,Follow-up,'--,'--,'--,'--,377,TCGA-D8-A1XC



follow_ups.timepoint_category
follow_ups.timepoint_category
'--                       6060
Follow-up                 2131
Last Contact              1095
Post Initial Treatment     141
Name: count, dtype: Int64

follow_ups.progression_or_recurrence_type
follow_ups.progression_or_recurrence_type
'--             9286
Distant           89
Locoregional      30
Unknown           22
Name: count, dtype: Int64


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

OLD_PATH = Path(
    "/content/gdrive/MyDrive/TCGA_BRCA/"
    "followup_processed/followup_processed.csv"
)

NEW_PATH = Path(
    "/content/gdrive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

# Check that both files exist.
print("Old file exists:", OLD_PATH.is_file())
print("New candidate exists:", NEW_PATH.is_file())

if OLD_PATH.is_file() and NEW_PATH.is_file():

    old = pd.read_csv(OLD_PATH)
    new = pd.read_csv(NEW_PATH)

    # Normalize patient IDs.
    for frame in [old, new]:
        frame["patient_id"] = (
            frame["patient_id"].astype(str).str.strip()
        )

    required = {"patient_id", "event", "survival_time"}

    assert required.issubset(old.columns), (
        f"Missing columns in old file: {required - set(old.columns)}"
    )
    assert required.issubset(new.columns), (
        f"Missing columns in new candidate: {required - set(new.columns)}"
    )

    print("\n--- FILE SIZES ---")
    print("Old file rows:", len(old))
    print("New candidate rows:", len(new))
    print("Old unique patients:", old["patient_id"].nunique())
    print("New unique patients:", new["patient_id"].nunique())

    print("\n--- DUPLICATE PATIENT IDS ---")
    print("Old:", old["patient_id"].duplicated().sum())
    print("New:", new["patient_id"].duplicated().sum())

    print("\n--- OLD EVENT COUNTS ---")
    print(old["event"].value_counts(dropna=False))

    print("\n--- NEW EVENT COUNTS ---")
    print(new["event"].value_counts(dropna=False))

    # Compare patients appearing in both files.
    comparison = old[
        ["patient_id", "event", "survival_time"]
    ].merge(
        new[["patient_id", "event", "survival_time"]],
        on="patient_id",
        how="outer",
        suffixes=("_old", "_new"),
        indicator=True,
        validate="one_to_one"
    )

    matched = comparison[comparison["_merge"] == "both"].copy()

    matched["event_changed"] = (
        matched["event_old"].astype(str)
        != matched["event_new"].astype(str)
    )

    old_time = pd.to_numeric(
        matched["survival_time_old"], errors="coerce"
    )
    new_time = pd.to_numeric(
        matched["survival_time_new"], errors="coerce"
    )

    matched["time_changed"] = ~np.isclose(
        old_time,
        new_time,
        rtol=0,
        atol=1e-8,
        equal_nan=True
    )

    print("\n--- PATIENT OVERLAP ---")
    print(comparison["_merge"].value_counts())

    print("\n--- DIFFERENCES AMONG MATCHED PATIENTS ---")
    print("Event label changed:", matched["event_changed"].sum())
    print("Survival time changed:", matched["time_changed"].sum())

    print("\n--- EXAMPLES WITH CHANGED LABELS OR TIMES ---")
    display(
        matched.loc[
            matched["event_changed"] | matched["time_changed"],
            [
                "patient_id",
                "event_old", "event_new",
                "survival_time_old", "survival_time_new",
                "event_changed", "time_changed"
            ]
        ].head(30)
    )

    print("\n--- PATIENTS PRESENT ONLY IN THE OLD FILE ---")
    display(
        comparison.loc[
            comparison["_merge"] == "left_only",
            ["patient_id", "event_old", "survival_time_old"]
        ].head(30)
    )

    print("\n--- PATIENTS PRESENT ONLY IN THE NEW CANDIDATE ---")
    display(
        comparison.loc[
            comparison["_merge"] == "right_only",
            ["patient_id", "event_new", "survival_time_new"]
        ].head(30)
    )

else:
    print(
        "\nA file is missing. Do not create or overwrite anything. "
        "Check the path before continuing."
    )

Old file exists: True
New candidate exists: True

--- FILE SIZES ---
Old file rows: 1095
New candidate rows: 1062
Old unique patients: 1095
New unique patients: 1062

--- DUPLICATE PATIENT IDS ---
Old: 0
New: 0

--- OLD EVENT COUNTS ---
event
0    999
1     96
Name: count, dtype: int64

--- NEW EVENT COUNTS ---
event
0    979
1     83
Name: count, dtype: int64

--- PATIENT OVERLAP ---
_merge
both          1062
left_only       33
right_only       0
Name: count, dtype: int64

--- DIFFERENCES AMONG MATCHED PATIENTS ---
Event label changed: 1062
Survival time changed: 80

--- EXAMPLES WITH CHANGED LABELS OR TIMES ---


,patient_id,event_old,event_new,survival_time_old,survival_time_new,event_changed,time_changed
0,TCGA-3C-AAAU,1,1.0,4047.0,1808.0,True,True
1,TCGA-3C-AALI,0,0.0,4005.0,4005.0,True,False
2,TCGA-3C-AALJ,0,0.0,1474.0,1474.0,True,False
3,TCGA-3C-AALK,0,0.0,1448.0,1448.0,True,False
4,TCGA-4H-AAAK,0,0.0,348.0,348.0,True,False
...,...,...,...,...,...,...,...
25,TCGA-A2-A04R,0,0.0,3709.0,3709.0,True,False
26,TCGA-A2-A04T,0,0.0,2246.0,2246.0,True,False
27,TCGA-A2-A04U,0,0.0,2654.0,2654.0,True,False
29,TCGA-A2-A04W,0,0.0,3102.0,3102.0,True,False



--- PATIENTS PRESENT ONLY IN THE OLD FILE ---


,patient_id,event_old,survival_time_old
28,TCGA-A2-A04V,1,1920.0
34,TCGA-A2-A0CM,1,754.0
68,TCGA-A2-A0SW,1,1365.0
69,TCGA-A2-A0SX,1,1534.0
103,TCGA-A2-A25D,0,1.0
...,...,...,...
676,TCGA-C8-A12Q,1,385.0
677,TCGA-C8-A12T,0,1.0
687,TCGA-C8-A133,0,1.0
705,TCGA-C8-A26Y,0,1.0



--- PATIENTS PRESENT ONLY IN THE NEW CANDIDATE ---


,patient_id,event_new,survival_time_new


In [ ]:
# READ-ONLY: Correct comparison of old and new outcome labels

import pandas as pd
import numpy as np

old = pd.read_csv(OLD_PATH)
new = pd.read_csv(NEW_PATH)

for df in [old, new]:
    df["patient_id"] = df["patient_id"].astype(str).str.strip()
    df["event"] = pd.to_numeric(df["event"], errors="coerce")
    df["survival_time"] = pd.to_numeric(
        df["survival_time"], errors="coerce"
    )

comparison = old[
    ["patient_id", "event", "survival_time"]
].merge(
    new[["patient_id", "event", "survival_time"]],
    on="patient_id",
    how="outer",
    suffixes=("_old", "_new"),
    indicator=True,
    validate="one_to_one"
)

matched = comparison[comparison["_merge"] == "both"].copy()

matched["event_changed"] = (
    matched["event_old"] != matched["event_new"]
).fillna(
    matched["event_old"].isna() != matched["event_new"].isna()
)

matched["time_changed"] = ~np.isclose(
    matched["survival_time_old"],
    matched["survival_time_new"],
    rtol=0,
    atol=1e-8,
    equal_nan=True
)

print("Matched patients:", len(matched))
print("Actual event-label changes:", matched["event_changed"].sum())
print("Survival-time changes:", matched["time_changed"].sum())

print("\nPatients with actual event-label changes:")
display(
    matched.loc[
        matched["event_changed"],
        [
            "patient_id",
            "event_old", "event_new",
            "survival_time_old", "survival_time_new"
        ]
    ].head(40)
)

print("\nPatients whose survival times changed:")
display(
    matched.loc[
        matched["time_changed"],
        [
            "patient_id",
            "event_old", "event_new",
            "survival_time_old", "survival_time_new"
        ]
    ].head(40)
)

print("\nOld censored patients with survival_time = 1:")
display(
    old.loc[
        (old["event"] == 0) & (old["survival_time"] == 1),
        ["patient_id", "event", "survival_time"]
    ].head(30)
)

Matched patients: 1062
Actual event-label changes: 0
Survival-time changes: 80

Patients with actual event-label changes:


,patient_id,event_old,event_new,survival_time_old,survival_time_new



Patients whose survival times changed:


,patient_id,event_old,event_new,survival_time_old,survival_time_new
0,TCGA-3C-AAAU,1,1.0,4047.0,1808.0
23,TCGA-A2-A04P,1,1.0,548.0,102.0
62,TCGA-A2-A0EW,1,1.0,1884.0,1353.0
79,TCGA-A2-A0YC,1,1.0,990.0,756.0
86,TCGA-A2-A0YJ,1,1.0,566.0,559.0
...,...,...,...,...,...
475,TCGA-B6-A0IK,1,1.0,571.0,489.0
477,TCGA-B6-A0IN,1,1.0,2573.0,1412.0
483,TCGA-B6-A0RH,1,1.0,6456.0,2165.0
484,TCGA-B6-A0RI,1,1.0,7126.0,1100.0



Old censored patients with survival_time = 1:


,patient_id,event,survival_time
103,TCGA-A2-A25D,0,1.0
165,TCGA-A8-A06N,0,1.0
193,TCGA-A8-A081,0,1.0
195,TCGA-A8-A083,0,1.0
204,TCGA-A8-A08H,0,1.0
...,...,...,...
705,TCGA-C8-A26Y,0,1.0
709,TCGA-C8-A275,0,1.0
740,TCGA-D8-A1JK,0,1.0
886,TCGA-E2-A9RU,0,1.0


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

OLD_PATH = (
    "/content/gdrive/MyDrive/TCGA_BRCA/"
    "followup_processed/followup_processed.csv"
)

NEW_PATH = (
    "/content/gdrive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audited_v2_candidate.csv"
)

RAW_PATH = (
    "/content/gdrive/MyDrive/BreastCanser-DataSet/follow_up.tsv"
)

OUT_DIR = Path(
    "/content/drive/MyDrive/BreastCanser-GenomicData/RFS_Endpoint"
)

old = pd.read_csv(OLD_PATH)
new = pd.read_csv(NEW_PATH)
raw = pd.read_csv(RAW_PATH, sep="\t", low_memory=False)

# Normalize patient IDs and outcome data.
for df in [old, new]:
    df["patient_id"] = (
        df["patient_id"].astype(str).str.strip()
    )
    df["event"] = pd.to_numeric(df["event"], errors="coerce")
    df["survival_time"] = pd.to_numeric(
        df["survival_time"], errors="coerce"
    )

# Identify shared patients with changed times and patients
# excluded from the new candidate.
comparison = old[
    ["patient_id", "event", "survival_time"]
].merge(
    new[["patient_id", "event", "survival_time"]],
    on="patient_id",
    how="outer",
    suffixes=("_old", "_new"),
    indicator=True,
    validate="one_to_one"
)

changed_times = comparison[
    (comparison["_merge"] == "both")
    & ~np.isclose(
        comparison["survival_time_old"],
        comparison["survival_time_new"],
        rtol=0,
        atol=1e-8,
        equal_nan=True
    )
].copy()

old_only_ids = comparison.loc[
    comparison["_merge"] == "left_only", "patient_id"
].tolist()

print("Changed survival times:", len(changed_times))
print("Patients only in old file:", len(old_only_ids))

# Normalize IDs in the raw follow-up records.
raw["patient_id"] = (
    raw["cases.submitter_id"]
    .astype(str)
    .str.strip()
    .str.replace(".", "-", regex=False)
)

# Keep only relevant raw fields that actually exist.
wanted = [
    "patient_id",
    "follow_ups.submitter_id",
    "follow_ups.timepoint_category",
    "follow_ups.progression_or_recurrence",
    "follow_ups.progression_or_recurrence_type",
    "follow_ups.days_to_recurrence",
    "follow_ups.days_to_progression",
    "follow_ups.days_to_follow_up",
]

raw_cols = [c for c in wanted if c in raw.columns]

# Include all raw records for the 80 patients with changed times.
changed_ids = set(changed_times["patient_id"])
changed_raw = raw.loc[
    raw["patient_id"].isin(changed_ids), raw_cols
].copy()

# Include all raw records for the 33 old-only patients.
old_only_raw = raw.loc[
    raw["patient_id"].isin(set(old_only_ids)), raw_cols
].copy()

# Save new audit files; original files remain untouched.
changed_times.to_csv(
    OUT_DIR / "final_audit_changed_survival_times.csv",
    index=False
)

changed_raw.to_csv(
    OUT_DIR / "final_audit_raw_records_changed_times.csv",
    index=False
)

old_only_raw.to_csv(
    OUT_DIR / "final_audit_raw_records_old_only_patients.csv",
    index=False
)

print("\nRaw records for changed-time patients:",
      len(changed_raw))
print("Raw records for old-only patients:",
      len(old_only_raw))

print("\nExamples: changed survival times")
display(changed_times.head(15))

print("\nRaw records for the first 5 changed-time patients")
display(
    changed_raw[
        changed_raw["patient_id"].isin(
            changed_times["patient_id"].head(5)
        )
    ]
)

print("\nRaw records for the first 5 old-only patients")
display(
    old_only_raw[
        old_only_raw["patient_id"].isin(old_only_ids[:5])
    ]
)


Changed survival times: 80
Patients only in old file: 33

Raw records for changed-time patients: 922
Raw records for old-only patients: 283

Examples: changed survival times


,patient_id,event_old,survival_time_old,event_new,survival_time_new,_merge
0,TCGA-3C-AAAU,1,4047.0,1.0,1808.0,both
23,TCGA-A2-A04P,1,548.0,1.0,102.0,both
62,TCGA-A2-A0EW,1,1884.0,1.0,1353.0,both
79,TCGA-A2-A0YC,1,990.0,1.0,756.0,both
86,TCGA-A2-A0YJ,1,566.0,1.0,559.0,both
101,TCGA-A2-A25B,1,1291.0,1.0,1290.0,both
104,TCGA-A2-A25E,1,3204.0,1.0,2989.0,both
108,TCGA-A2-A3XS,1,1032.0,1.0,682.0,both
110,TCGA-A2-A3XU,1,912.0,1.0,574.0,both
114,TCGA-A2-A3XY,1,1093.0,1.0,802.0,both



Raw records for the first 5 changed-time patients


,patient_id,follow_ups.submitter_id,follow_ups.timepoint_category,follow_ups.progression_or_recurrence,follow_ups.progression_or_recurrence_type,follow_ups.days_to_recurrence,follow_ups.days_to_progression,follow_ups.days_to_follow_up
4065,TCGA-3C-AAAU,TCGA-3C-AAAU_follow_up3,Follow-up,'--,'--,'--,'--,3928
4066,TCGA-3C-AAAU,TCGA-3C-AAAU_follow_up4,Follow-up,'--,'--,'--,'--,4047
4067,TCGA-3C-AAAU,TCGA-3C-AAAU_follow_up,Last Contact,'--,'--,'--,'--,4047
4068,TCGA-3C-AAAU,TCGA-3C-AAAU_follow_up2,Follow-up,'--,'--,'--,'--,3767
4069,TCGA-3C-AAAU,TCGA-3C-AAAU_follow_up5,Post Initial Treatment,Yes,Distant,'--,1808,1808
...,...,...,...,...,...,...,...,...
7577,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--
7578,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--
7579,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--
7580,TCGA-A2-A04P,'--,'--,'--,'--,'--,'--,'--



Raw records for the first 5 old-only patients


,patient_id,follow_ups.submitter_id,follow_ups.timepoint_category,follow_ups.progression_or_recurrence,follow_ups.progression_or_recurrence_type,follow_ups.days_to_recurrence,follow_ups.days_to_progression,follow_ups.days_to_follow_up
956,TCGA-A2-A0SX,TCGA-A2-A0SX_follow_up3,Follow-up,'--,'--,'--,'--,1534
957,TCGA-A2-A0SX,TCGA-A2-A0SX_follow_up,Last Contact,'--,'--,'--,'--,1534
958,TCGA-A2-A0SX,TCGA-A2-A0SX_follow_up2,Follow-up,'--,'--,'--,'--,1288
959,TCGA-A2-A0SX,TCGA-A2-A0SX_follow_up4,Post Initial Treatment,Yes,Unknown,'--,'--,1239
960,TCGA-A2-A0SX,'--,'--,'--,'--,'--,'--,'--
...,...,...,...,...,...,...,...,...
8736,TCGA-A2-A0CM,'--,'--,'--,'--,'--,'--,'--
8737,TCGA-A2-A0CM,'--,'--,'--,'--,'--,'--,'--
8738,TCGA-A2-A0CM,'--,'--,'--,'--,'--,'--,'--
8739,TCGA-A2-A0CM,'--,'--,'--,'--,'--,'--,'--


In [ ]:

# READ-ONLY: inspect all patients present only in the old file

old_only = old.loc[
    old["patient_id"].isin(old_only_ids),
    ["patient_id", "event", "survival_time"]
].copy()

print("Old-only patients:", len(old_only))
display(old_only.sort_values("patient_id"))

# Display only source records with at least one meaningful
# value in the relevant follow-up fields.
relevant_cols = [
    "follow_ups.timepoint_category",
    "follow_ups.progression_or_recurrence",
    "follow_ups.progression_or_recurrence_type",
    "follow_ups.days_to_recurrence",
    "follow_ups.days_to_progression",
    "follow_ups.days_to_follow_up",
]

available_cols = [
    c for c in relevant_cols if c in raw.columns
]

source = raw.loc[
    raw["patient_id"].isin(old_only_ids),
    ["patient_id", "follow_ups.submitter_id"] + available_cols
].copy()

def meaningful(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() not in {
        "", "--", "nan", "none", "null"
    }

has_data = source[available_cols].apply(
    lambda col: col.map(meaningful)
).any(axis=1)

source = source.loc[has_data].sort_values(
    ["patient_id", "follow_ups.submitter_id"]
)

print("Meaningful source records:", len(source))
display(source.to_string(index=False))


Old-only patients: 33


,patient_id,event,survival_time
28,TCGA-A2-A04V,1,1920.0
34,TCGA-A2-A0CM,1,754.0
68,TCGA-A2-A0SW,1,1365.0
69,TCGA-A2-A0SX,1,1534.0
103,TCGA-A2-A25D,0,1.0
...,...,...,...
705,TCGA-C8-A26Y,0,1.0
740,TCGA-D8-A1JK,0,1.0
755,TCGA-D8-A1XC,1,377.0
886,TCGA-E2-A9RU,0,1.0


Meaningful source records: 283


"  patient_id follow_ups.submitter_id follow_ups.timepoint_category follow_ups.progression_or_recurrence follow_ups.progression_or_recurrence_type follow_ups.days_to_recurrence follow_ups.days_to_progression follow_ups.days_to_follow_up\nTCGA-A2-A04V                     '--                           '--                                  '--                                       '--                           '--                            '--                          '--\nTCGA-A2-A04V                     '--                           '--                                  '--                                       '--                           '--                            '--                          '--\nTCGA-A2-A04V                     '--                           '--                                  '--                                       '--                           '--                            '--                          '--\nTCGA-A2-A04V                     '--               

In [ ]:

# READ-ONLY diagnostic: correctly recognize quoted placeholders

def clean_for_audit(value):
    if pd.isna(value):
        return ""
    return str(value).strip().strip("'\"").strip()

audit_cols = [
    "follow_ups.timepoint_category",
    "follow_ups.progression_or_recurrence",
    "follow_ups.progression_or_recurrence_type",
    "follow_ups.days_to_recurrence",
    "follow_ups.days_to_progression",
    "follow_ups.days_to_follow_up",
]

audit_cols = [c for c in audit_cols if c in raw.columns]

source = raw.loc[
    raw["patient_id"].isin(old_only_ids),
    ["patient_id", "follow_ups.submitter_id"] + audit_cols
].copy()

# Normalize placeholders in the diagnostic copy only.
for col in audit_cols:
    source[col] = source[col].map(clean_for_audit)

missing_values = {"", "--", "nan", "none", "null"}

has_real_value = source[audit_cols].apply(
    lambda col: col.map(
        lambda x: str(x).lower() not in missing_values
    )
).any(axis=1)

meaningful_source = source.loc[has_real_value].copy()

print("Old-only patients:", len(old_only_ids))
print(
    "Patients with at least one meaningful source record:",
    meaningful_source["patient_id"].nunique()
)
print("Meaningful source records:", len(meaningful_source))

# Show every meaningful record for the old-only patients.
display(
    meaningful_source.sort_values(
        ["patient_id", "follow_ups.submitter_id"]
    ).to_string(index=False)
)


Old-only patients: 33
Patients with at least one meaningful source record: 33
Meaningful source records: 94


'  patient_id follow_ups.submitter_id follow_ups.timepoint_category follow_ups.progression_or_recurrence follow_ups.progression_or_recurrence_type follow_ups.days_to_recurrence follow_ups.days_to_progression follow_ups.days_to_follow_up\nTCGA-A2-A04V  TCGA-A2-A04V_follow_up                  Last Contact                                   --                                        --                            --                             --                         1920\nTCGA-A2-A04V TCGA-A2-A04V_follow_up2                     Follow-up                                   --                                        --                            --                             --                         1920\nTCGA-A2-A04V TCGA-A2-A04V_follow_up3        Post Initial Treatment                                  Yes                                   Unknown                            --                             --                         1470\nTCGA-A2-A0CM  TCGA-A2-A0CM_follow_up               

In [ ]:

# Final audit of the 33 patients missing from the new candidate.
# Read-only: does not alter either outcome file.

audit = raw.loc[
    raw["patient_id"].isin(old_only_ids)
].copy()

def clean_value(value):
    if pd.isna(value):
        return ""
    return str(value).strip().strip("'\"").strip()

def parse_positive(series):
    values = pd.to_numeric(
        series.map(clean_value).replace("", np.nan),
        errors="coerce"
    )
    return values.where(values > 0)

for col in [
    "follow_ups.days_to_recurrence",
    "follow_ups.days_to_progression",
    "follow_ups.days_to_follow_up",
]:
    audit[col + "_num"] = parse_positive(audit[col])

audit["_flag"] = (
    audit["follow_ups.progression_or_recurrence"]
    .map(clean_value)
    .str.lower()
)

rows = []

for pid, group in audit.groupby("patient_id"):
    old_row = old.loc[old["patient_id"] == pid].iloc[0]

    rec = group["follow_ups.days_to_recurrence_num"].dropna()
    prog = group["follow_ups.days_to_progression_num"].dropna()
    fu = group["follow_ups.days_to_follow_up_num"].dropna()

    event_times = pd.concat([rec, prog]).dropna()
    yes_exists = group["_flag"].isin(
        ["yes", "true", "1"]
    ).any()

    rows.append({
        "patient_id": pid,
        "old_event": old_row["event"],
        "old_time": old_row["survival_time"],
        "raw_yes_flag": yes_exists,
        "recurrence_times": sorted(rec.unique().tolist()),
        "progression_times": sorted(prog.unique().tolist()),
        "earliest_recorded_event_time": (
            event_times.min() if len(event_times) else np.nan
        ),
        "followup_times": sorted(fu.unique().tolist()),
        "max_followup_time": (
            fu.max() if len(fu) else np.nan
        ),
        "candidate_status": (
            "Event time available in raw fields"
            if len(event_times) > 0
            else (
                "Event reported, event time missing"
                if yes_exists
                else (
                    "No event time; follow-up available"
                    if len(fu) > 0
                    else "No usable event or follow-up time"
                )
            )
        )
    })

summary = pd.DataFrame(rows).sort_values("patient_id")

print("Patients audited:", len(summary))
print("\nCounts by preliminary audit category:")
display(summary["candidate_status"].value_counts().to_frame("patients"))

print("\nPatient-level audit:")
display(summary.to_string(index=False))

summary.to_csv(
    OUT_DIR / "final_audit_33_excluded_patients.csv",
    index=False
)
print("\nSaved the audit summary. Neither outcome file was changed.")


Patients audited: 33

Counts by preliminary audit category:


,patients
candidate_status,
No usable event or follow-up time,20
"Event reported, event time missing",13



Patient-level audit:


'  patient_id  old_event  old_time  raw_yes_flag recurrence_times progression_times  earliest_recorded_event_time           followup_times  max_followup_time                   candidate_status\nTCGA-A2-A04V          1    1920.0          True               []                []                           NaN         [1470.0, 1920.0]             1920.0 Event reported, event time missing\nTCGA-A2-A0CM          1     754.0          True               []                []                           NaN           [382.0, 754.0]              754.0 Event reported, event time missing\nTCGA-A2-A0SW          1    1365.0          True               []                []                           NaN          [653.0, 1365.0]             1365.0 Event reported, event time missing\nTCGA-A2-A0SX          1    1534.0          True               []                []                           NaN [1239.0, 1288.0, 1534.0]             1534.0 Event reported, event time missing\nTCGA-A2-A25D          0       1.0 


Saved the audit summary. Neither outcome file was changed.


In [ ]:

# FINAL RECONCILIATION — read-only

raw_ids = set(raw["patient_id"].dropna().astype(str).str.strip())
candidate_ids = set(
    new["patient_id"].dropna().astype(str).str.strip()
)

print("Unique raw patients:", len(raw_ids))
print("Patients in new candidate:", len(candidate_ids))
print("Raw patients absent from candidate:", len(raw_ids - candidate_ids))
print("Candidate IDs not found in raw data:", len(candidate_ids - raw_ids))

# Check whether every absent raw patient has an audit entry.
audit_all_path = (
    "/content/gdrive/MyDrive/BreastCanser-GenomicData/"
    "RFS_Endpoint/followup_rfs_audit_all_patients.csv"
)

audit_all = pd.read_csv(audit_all_path)
print("Columns in full patient audit:", audit_all.columns.tolist())

# Identify the patient-ID column in the audit.
possible_id_cols = [
    c for c in audit_all.columns
    if c.lower() in {"patient_id", "cases.submitter_id"}
]
print("Possible audit ID columns:", possible_id_cols)

if possible_id_cols:
    audit_ids = set(
        audit_all[possible_id_cols[0]]
        .dropna().astype(str).str.strip()
        .str.replace(".", "-", regex=False)
    )
    print("Raw patients absent from candidate and present in audit:",
          len((raw_ids - candidate_ids) & audit_ids))
    print("Raw patients absent from candidate and missing from audit:",
          len((raw_ids - candidate_ids) - audit_ids))


Unique raw patients: 1096
Patients in new candidate: 1062
Raw patients absent from candidate: 34
Candidate IDs not found in raw data: 0
Columns in full patient audit: ['patient_id', 'event', 'survival_time', 'endpoint_status']
Possible audit ID columns: ['patient_id']
Raw patients absent from candidate and present in audit: 34
Raw patients absent from candidate and missing from audit: 0
